In [ ]:
import os
import re
import requests
import fitz
import numpy as np
import pandas as pd
import faiss
import torch

from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer

PDF_URL = "https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.100-1.pdf"
PDF_PATH = "nist_ai_rmf.pdf"

response = requests.get(PDF_URL, timeout=60)
response.raise_for_status()

with open(PDF_PATH, "wb") as f:
    f.write(response.content)

print("PDF downloaded successfully!")
print("File size:", round(os.path.getsize(PDF_PATH) / (1024 * 1024), 2), "MB")

In [38]:
doc = fitz.open(PDF_PATH)

pages = []

for page_number, page in enumerate(doc):
    text = page.get_text("text")

    if text.strip():
        pages.append({
            "page": page_number + 1,
            "text": text.strip()
        })

print("Total PDF pages:", len(doc))
print("Pages with extracted text:", len(pages))

assert len(doc) >= 30, "Document must contain at least 30 pages."
assert len(pages) >= 30, "Insufficient extractable pages."

print("\nSample extracted text:\n")
print(pages[5]["text"][:1000])

Total PDF pages: 48
Pages with extracted text: 48

Sample extracted text:

NIST AI 100-1
AI RMF 1.0
Executive Summary
Artificial intelligence (AI) technologies have significant potential to transform society and
people’s lives – from commerce and health to transportation and cybersecurity to the envi-
ronment and our planet. AI technologies can drive inclusive economic growth and support
scientific advancements that improve the conditions of our world. AI technologies, how-
ever, also pose risks that can negatively impact individuals, groups, organizations, commu-
nities, society, the environment, and the planet. Like risks for other types of technology, AI
risks can emerge in a variety of ways and can be characterized as long- or short-term, high-
or low-probability, systemic or localized, and high- or low-impact.
The AI RMF refers to an AI system as an engineered or machine-based system that
can, for a given set of objectives, generate outputs such as predictions, recommenda-
tions, 

In [39]:
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(MODEL_NAME)

EMBEDDING_DIM = embedding_model.get_sentence_embedding_dimension()

print("Embedding model loaded!")
print("Embedding dimension:", EMBEDDING_DIM)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1541.52it/s]


Embedding model loaded!
Embedding dimension: 384


C:\Users\CH Ganesh\AppData\Local\Temp\ipykernel_11736\1316359113.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMBEDDING_DIM = embedding_model.get_sentence_embedding_dimension()


In [40]:
tokenizer = embedding_model.tokenizer

def clean_text(text):
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'\x00', '', text)
    return text.strip()

def create_chunks(pages, chunk_size, overlap=50):
    # The MiniLM encoder supports at most 512 tokens per input; keep the 512-token
    # experiment intact instead of silently truncating it at the default 256.
    embedding_model.max_seq_length = 512

    chunks = []

    for page in pages:

        text = clean_text(page["text"])

        tokens = tokenizer.encode(
            text,
            add_special_tokens=False
        )

        start = 0

        while start < len(tokens):

            end = min(start + chunk_size, len(tokens))

            chunk_tokens = tokens[start:end]

            chunk_text = tokenizer.decode(
                chunk_tokens,
                skip_special_tokens=True
            )

            if len(chunk_text.strip()) > 30:

                chunks.append({
                    "text": chunk_text,
                    "page": page["page"],
                    "start_token": start,
                    "end_token": end
                })

            if end == len(tokens):
                break

            start += chunk_size - overlap

    return chunks

small_chunks = create_chunks(
    pages,
    chunk_size=256,
    overlap=50
)

large_chunks = create_chunks(
    pages,
    chunk_size=512,
    overlap=50
)

print("256-token chunks:", len(small_chunks))
print("512-token chunks:", len(large_chunks))

print("\nSample chunk:\n")
print(small_chunks[0]["text"])

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (624 > 512). Running this sequence through the model will result in indexing errors


256-token chunks: 118
512-token chunks: 68

Sample chunk:

nist ai 100 - 1 artificial intelligence risk management framework ( ai rmf 1. 0 )


In [41]:
def generate_embeddings(chunks):

    texts = [chunk["text"] for chunk in chunks]

    embeddings = embedding_model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    return np.asarray(embeddings, dtype=np.float32)

small_embeddings = generate_embeddings(small_chunks)

large_embeddings = generate_embeddings(large_chunks)

print("Small embeddings:", small_embeddings.shape)
print("Large embeddings:", large_embeddings.shape)

Batches: 100%|██████████| 3/3 [00:06<00:00,  2.32s/it]

Small embeddings: (118, 384)
Large embeddings: (68, 384)


In [42]:
def build_faiss_index(embeddings):

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatIP(dimension)

    index.add(embeddings)

    return index

small_index = build_faiss_index(small_embeddings)

large_index = build_faiss_index(large_embeddings)

print("FAISS indexes created successfully!")

print("Small index vectors:", small_index.ntotal)
print("Large index vectors:", large_index.ntotal)

FAISS indexes created successfully!
Small index vectors: 118
Large index vectors: 68


In [43]:
def retrieve(query, index, chunks, top_k=5):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)

    scores, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx == -1:
            continue

        results.append({
            "text": chunks[idx]["text"],
            "page": chunks[idx]["page"],
            "score": float(score),
            "chunk_id": int(idx)
        })

    return results

# Test retrieval

test_query = "What are the characteristics of trustworthy AI systems?"

results = retrieve(
    test_query,
    small_index,
    small_chunks,
    top_k=5
)

for i, result in enumerate(results, 1):

    print(f"\nResult {i}")
    print("Page:", result["page"])
    print("Similarity:", round(result["score"], 4))
    print(result["text"][:500])
    print("-" * 80)


Result 1
Page: 17
Similarity: 0.7841
nist ai 100 - 1 ai rmf 1. 0 3. ai risks and trustworthiness for ai systems to be trustworthy, they often need to be responsive to a multiplicity of cri - teria that are of value to interested parties. approaches which enhance ai trustworthiness can reduce negative ai risks. this framework articulates the following characteristics of trustworthy ai and offers guidance for addressing them. characteristics of trustworthy ai systems include : valid and reliable, safe, secure and resilient, accountab
--------------------------------------------------------------------------------

Result 2
Page: 18
Similarity: 0.7356
social contexts are managed appropriately. understanding and treatment of trustworthiness characteristics depends on an ai actor ’ s particular role within the ai lifecycle. for any given ai system, an ai designer or developer may have a different perception of the characteristics than the deployer. trustworthiness characteristics explained

In [ ]:
# Load the authored evaluation set from the separate submission CSV.
question_file = "15_test_questions_and_source_passages.csv"
if not os.path.exists(question_file):
    question_file = os.path.join("RAG(Task_2)", question_file)
questions_df = pd.read_csv(question_file)
test_questions = [
    {
        "question": row["Question"],
        "gold_page": int(row["Correct Source Page (PDF)"]),
        "gold_printed_page": int(row["Correct Source Page (printed)" ]),
        "gold_passage": row["Correct Source Passage"]
    }
    for _, row in questions_df.iterrows()
]
print("Loaded evaluation questions:", len(test_questions))


## Evaluation notes and page numbering

The PDF contains front matter before its printed page 1. The `pages` list numbers physical PDF pages from 1, so the printed page number is five less than the physical PDF page for this document. Each test question stores both labels; retrieval is evaluated against the physical PDF page attached to each retrieved chunk. Recall@5 is the fraction of the 15 questions for which that source page appears among the top five chunks. Use the current run's printed metrics below; saved outputs from earlier versions have been removed because they used the wrong page labels.


In [45]:
def evaluate_retrieval(test_questions, index, chunks, top_k=5):
    evaluation_results = []
    for item in test_questions:
        results = retrieve(item["question"], index, chunks, top_k=top_k)
        retrieved_pages = [r["page"] for r in results]
        gold_page = item["gold_page"]  # 1-based physical PDF page
        hit = int(gold_page in retrieved_pages)
        evaluation_results.append({
            "Question": item["question"],
            "Gold PDF Page": gold_page,
            "Gold Printed Page": item["gold_printed_page"],
            "Retrieved Pages": retrieved_pages,
            "Hit@5": hit,
            "Best Similarity": results[0]["score"] if results else 0
        })
    return pd.DataFrame(evaluation_results)

# Evaluate 256-token chunks
small_eval = evaluate_retrieval(
    test_questions,
    small_index,
    small_chunks,
    top_k=5
)

# Evaluate 512-token chunks
large_eval = evaluate_retrieval(
    test_questions,
    large_index,
    large_chunks,
    top_k=5
)

# Recall@5 is the fraction of questions whose gold PDF page appears in the top five retrieved chunks.
# The printed page and PDF page are both retained to avoid front-matter offset errors.
# Calculate Recall@5
small_recall = small_eval["Hit@5"].mean()
large_recall = large_eval["Hit@5"].mean()

print("256-token Recall@5:", round(small_recall * 100, 2), "%")
print("512-token Recall@5:", round(large_recall * 100, 2), "%")

# Display results
display(small_eval)
display(large_eval)

256-token Recall@5: 66.67 %
512-token Recall@5: 53.33 %


,Question,Gold PDF Page,Gold Printed Page,Retrieved Pages,Hit@5,Best Similarity
0,What is the primary purpose of the NIST AI Ris...,7,2,"[1, 2, 37, 6, 8]",0,0.762886
1,What are the four core functions of the AI RMF?,25,20,"[25, 8, 38, 26, 47]",1,0.762483
2,What is the purpose of the GOVERN function?,26,21,"[26, 27, 28, 28, 29]",1,0.477899
3,What is the purpose of the MAP function?,29,24,"[30, 46, 29, 30, 30]",1,0.555212
4,What is the purpose of the MEASURE function?,33,28,"[33, 30, 26, 35, 33]",1,0.357324
5,What is the purpose of the MANAGE function?,36,31,"[26, 37, 36, 37, 27]",1,0.497709
6,What are the characteristics of trustworthy AI...,17,12,"[17, 18, 17, 5, 5]",1,0.784124
7,Why is transparency important in AI systems?,20,15,"[20, 21, 22, 21, 20]",1,0.736364
8,What does the AI RMF say accountability presup...,20,15,"[47, 24, 8, 7, 7]",0,0.634842
9,What does validation confirm for a specific AI...,18,13,"[18, 19, 40, 19, 18]",1,0.626511


,Question,Gold PDF Page,Gold Printed Page,Retrieved Pages,Hit@5,Best Similarity
0,What is the primary purpose of the NIST AI Ris...,7,2,"[1, 2, 38, 25, 8]",0,0.762885
1,What are the four core functions of the AI RMF?,25,20,"[25, 8, 37, 26, 46]",1,0.762594
2,What is the purpose of the GOVERN function?,26,21,"[27, 29, 25, 28, 46]",0,0.352170
3,What is the purpose of the MAP function?,29,24,"[30, 46, 29, 32, 25]",1,0.358917
4,What is the purpose of the MEASURE function?,33,28,"[33, 33, 34, 25, 38]",1,0.443845
5,What is the purpose of the MANAGE function?,36,31,"[37, 25, 26, 27, 46]",0,0.425089
6,What are the characteristics of trustworthy AI...,17,12,"[17, 7, 5, 18, 19]",1,0.737151
7,Why is transparency important in AI systems?,20,15,"[20, 21, 21, 22, 45]",1,0.743004
8,What does the AI RMF say accountability presup...,20,15,"[25, 8, 7, 4, 24]",0,0.586161
9,What does validation confirm for a specific AI...,18,13,"[40, 19, 18, 16, 7]",1,0.564686


## BM25 and re-ranking (in my own words)

BM25 adds keyword matching to embedding search. It is useful when a question contains exact names, acronyms, or rare terms that a semantic embedding may not match closely. A hybrid retriever can combine BM25 and vector results.

A re-ranker takes the query and retrieved passages together, scores how relevant each passage is to that query, and reorders the candidates. It can improve the ordering of a shortlist, but it does not recover a passage that the first-stage retriever never returned.


In [46]:
# Display misses for both chunk sizes. Cell below prints two concrete cases.
small_failures = small_eval[small_eval["Hit@5"] == 0]
large_failures = large_eval[large_eval["Hit@5"] == 0]

print("FAILURES WITH 256-TOKEN CHUNKS")
display(small_failures)
print("FAILURES WITH 512-TOKEN CHUNKS")
display(large_failures)


FAILURES WITH 256-TOKEN CHUNKS


,Question,Gold PDF Page,Gold Printed Page,Retrieved Pages,Hit@5,Best Similarity
0,What is the primary purpose of the NIST AI Ris...,7,2,"[1, 2, 37, 6, 8]",0,0.762886
8,What does the AI RMF say accountability presup...,20,15,"[47, 24, 8, 7, 7]",0,0.634842
10,How does the AI RMF define a safe AI system?,19,14,"[25, 8, 6, 20, 35]",0,0.700656
12,Where does the AI RMF say human judgment shoul...,17,12,"[45, 47, 14, 7, 9]",0,0.602769
14,How should risk management be carried out over...,25,20,"[26, 6, 13, 13, 9]",0,0.782337


FAILURES WITH 512-TOKEN CHUNKS


,Question,Gold PDF Page,Gold Printed Page,Retrieved Pages,Hit@5,Best Similarity
0,What is the primary purpose of the NIST AI Ris...,7,2,"[1, 2, 38, 25, 8]",0,0.762885
2,What is the purpose of the GOVERN function?,26,21,"[27, 29, 25, 28, 46]",0,0.352170
5,What is the purpose of the MANAGE function?,36,31,"[37, 25, 26, 27, 46]",0,0.425089
8,What does the AI RMF say accountability presup...,20,15,"[25, 8, 7, 4, 24]",0,0.586161
10,How does the AI RMF define a safe AI system?,19,14,"[25, 13, 8, 6, 35]",0,0.702234
12,Where does the AI RMF say human judgment shoul...,17,12,"[7, 14, 45, 23, 45]",0,0.553535
14,How should risk management be carried out over...,25,20,"[6, 12, 38, 19, 10]",0,0.758924


In [47]:
# Select two actual misses and show their expected passage and retrieved candidates.
misses = small_eval[small_eval["Hit@5"] == 0].head(2)
for _, row in misses.iterrows():
    item = next(q for q in test_questions if q["question"] == row["Question"])
    print("\nQuestion:", item["question"])
    print("Expected source passage:", item["gold_passage"])
    print("Expected source pages: printed", item["gold_printed_page"],
          "| physical PDF", item["gold_page"])
    print("Top-five retrieved PDF pages:", row["Retrieved Pages"])
    print("Fix: add BM25 candidates for exact terms, retrieve a larger shortlist, then re-rank it.")



Question: What is the primary purpose of the NIST AI Risk Management Framework?
Expected source passage: The goal of the AI RMF is to offer a resource to organizations designing, developing, deploying, or using AI systems to help manage AI risks and promote trustworthy and responsible development and use of AI systems.
Expected source pages: printed 2 | physical PDF 7
Top-five retrieved PDF pages: [1, 2, 37, 6, 8]
Fix: add BM25 candidates for exact terms, retrieve a larger shortlist, then re-rank it.

Question: What does the AI RMF say accountability presupposes?
Expected source passage: Trustworthy AI depends upon accountability. Accountability presupposes transparency.
Expected source pages: printed 15 | physical PDF 20
Top-five retrieved PDF pages: [47, 24, 8, 7, 7]
Fix: add BM25 candidates for exact terms, retrieve a larger shortlist, then re-rank it.


## Two retrieval failures and fixes

The code below selects two misses from the current 256-token evaluation and prints each question, its annotated source passage, the expected printed and PDF pages, and the pages actually retrieved. These are the two reported failure cases for this run. If fewer than two misses remain after rerunning, report the number honestly and inspect a second query or chunk-size run before claiming two failures. A practical fix for exact function names is BM25 candidate retrieval; retrieve a larger shortlist and re-rank it so the source passage can move above loosely related passages.


In [48]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

model_id = "google/flan-t5-base"

# Load the tokenizer and model directly
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"LLM loaded successfully on {device}!")

Loading weights: 100%|██████████| 282/282 [00:02<00:00, 114.00it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


LLM loaded successfully on cpu!


In [49]:
def generate_answer(question, index, chunks, top_k=5):
    retrieved_results = retrieve(question, index, chunks, top_k=top_k)

    # FLAN-T5-base has a 512-token input limit. Budget context explicitly so
    # the question and answer instruction are not truncated off the prompt.
    context_parts = []
    remaining_tokens = 360
    for result in retrieved_results:
        passage_ids = tokenizer.encode(result["text"], add_special_tokens=False)
        passage_ids = passage_ids[:min(110, remaining_tokens)]
        if not passage_ids:
            continue
        passage = tokenizer.decode(passage_ids, skip_special_tokens=True)
        context_parts.append(f"[Source: PDF page {result['page']}]\n{passage}")
        remaining_tokens -= len(passage_ids)
        if remaining_tokens <= 0:
            break
    context = "\n\n".join(context_parts)

    prompt = f"""Answer the question using only the context. Give a concise answer and cite its PDF page. If the context does not answer it, say that the document does not contain sufficient information.

Context:
{context}

Question: {question}
Answer:"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    return {
        "question": question,
        "answer": generated_text,
        "sources": [
            {"page": result["page"], "score": round(result["score"], 4)}
            for result in retrieved_results
        ]
    }


In [52]:
import pandas as pd

# 1. Export 15 questions and their correct source passages.
source_passages_df = pd.DataFrame([
    {
        "Question": item["question"],
        "Correct Source Page (PDF)": item["gold_page"],
        "Correct Source Page (printed)": item["gold_printed_page"],
        "Correct Source Passage": item["gold_passage"]
    }
    for item in test_questions
])

source_passages_df.to_csv(
    "15_test_questions_and_source_passages.csv",
    index=False
)

# 2. Reconstruct and export chunk comparison
comparison = pd.DataFrame({
    "Chunk Size": [256, 512],
    "Recall@5": [small_recall, large_recall]
})

comparison.to_csv(
    "chunk_size_comparison.csv",
    index=False
)

# 3. Reconstruct and export combined retrieval results
small_eval_df = small_eval.copy()
small_eval_df["Chunk Size"] = 256
large_eval_df = large_eval.copy()
large_eval_df["Chunk Size"] = 512

final_evaluation = pd.concat([small_eval_df, large_eval_df], ignore_index=True)

final_evaluation.to_csv(
    "retrieval_evaluation.csv",
    index=False
)

# 4. Build and export up to two actual retrieval misses across both chunk sizes.
miss_rows = pd.concat([
    small_eval.assign(**{"Chunk Size": 256}),
    large_eval.assign(**{"Chunk Size": 512})
], ignore_index=True)
miss_rows = miss_rows[miss_rows["Hit@5"] == 0].drop_duplicates(
    subset=["Question"], keep="first"
).head(2)
question_lookup = {item["question"]: item for item in test_questions}
failure_analysis = pd.DataFrame([
    {
        "Question": row["Question"],
        "Chunk Size": row["Chunk Size"],
        "Correct Source Page (printed)": question_lookup[row["Question"]]["gold_printed_page"],
        "Correct Source Page (PDF)": question_lookup[row["Question"]]["gold_page"],
        "Correct Source Passage": question_lookup[row["Question"]]["gold_passage"],
        "Retrieved Pages": row["Retrieved Pages"],
        "Reason": "The annotated source page was absent from the top five retrieved chunks.",
        "Possible Fix": "Add BM25 candidates for exact terms, retrieve a larger shortlist, then re-rank it."
    }
    for _, row in miss_rows.iterrows()
])
failure_analysis.to_csv("retrieval_failure_analysis.csv", index=False)

print("All evaluation files exported successfully!")

All evaluation files exported successfully!


In [53]:
# Programmatically define the best chunk size based on recall performance
best_chunk_size = (256 if small_recall > large_recall else
                   512 if large_recall > small_recall else None)

print("=" * 70)
print("RAG FROM SCRATCH - EXPERIMENT SUMMARY")
print("=" * 70)

print("\nDOCUMENT")
print("NIST AI Risk Management Framework 1.0")
print("Total PDF pages:", len(doc))

print("\nEMBEDDING MODEL")
print(MODEL_NAME)

print("\nVECTOR DATABASE")
print("FAISS - IndexFlatIP")
print("Similarity: Cosine Similarity")

print("\nCHUNKING CONFIGURATION")
print("Small chunk size: 256 tokens")
print("Large chunk size: 512 tokens")
print("Overlap: 50 tokens")

print("\nEVALUATION")
print("Number of test questions:", len(test_questions))
print(f"256-token Recall@5: {small_recall * 100:.2f}%")
print(f"512-token Recall@5: {large_recall * 100:.2f}%")

print("\nBEST CONFIGURATION")
if best_chunk_size is None:
    print("Chunk size: tie; neither configuration performed better")
else:
    print(f"Chunk size: {best_chunk_size} tokens")

if small_recall > large_recall:
    print("Why 256 tokens worked better: its focused chunks matched the gold pages more often")
    print("in this test set. Smaller chunks can reduce unrelated context in each embedding.")
elif large_recall > small_recall:
    print("Why 512 tokens worked better: its broader context matched the gold pages more often")
    print("in this test set. Larger chunks can keep definitions and nearby context together.")
else:
    print("Why neither worked better: both chunk sizes retrieved the same number of gold pages")
    print("in this test set, so there is no measured winner. The tradeoff is focus vs context.")

print("\nFILES GENERATED")
print("- 15_test_questions_and_source_passages.csv")
print("- chunk_size_comparison.csv")
print("- retrieval_evaluation.csv")
print("- retrieval_failure_analysis.csv")

print("\n" + "=" * 70)

RAG FROM SCRATCH - EXPERIMENT SUMMARY

DOCUMENT
NIST AI Risk Management Framework 1.0
Total PDF pages: 48

EMBEDDING MODEL
sentence-transformers/all-MiniLM-L6-v2

VECTOR DATABASE
FAISS - IndexFlatIP
Similarity: Cosine Similarity

CHUNKING CONFIGURATION
Small chunk size: 256 tokens
Large chunk size: 512 tokens
Overlap: 50 tokens

EVALUATION
Number of test questions: 15
256-token Recall@5: 66.67%
512-token Recall@5: 53.33%

BEST CONFIGURATION
Chunk size: 256 tokens
Why 256 tokens worked better: its focused chunks matched the gold pages more often
in this test set. Smaller chunks can reduce unrelated context in each embedding.

FILES GENERATED
- 15_test_questions_and_source_passages.csv
- chunk_size_comparison.csv
- retrieval_evaluation.csv
- retrieval_failure_analysis.csv

